# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — ten decision dates, one shared module

From here on every notebook imports `work/scripts/search_queue.py`, so the frame, the labels, the
frozen ML-07 rule and the metrics are defined once. The module has its own test suite,
`work/tests/test_search_queue.py`, which runs in section 3.

The ML-04 setup (decision date T, the 14 days before it for features, the 14 days from it for the
label) is repeated at **ten decision dates**: every second Monday from 2026-01-05 to 2026-05-11.
Every outcome window ends by 2026-05-24, so **June 2026 stays sealed**. The setup cell asserts it.

In [1]:
import os, sys, json, subprocess, importlib.util, urllib.request, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import numpy as np
import pandas as pd
import sklearn

# the shared module: from the repo when run inside it, else fetched from the repo (Colab opened from GitHub)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT / 'work' / 'scripts'))
else:
    ROOT = Path.cwd()
    urllib.request.urlretrieve('https://raw.githubusercontent.com/WingManOO7/ML-INTER/main/work/scripts/search_queue.py',
                               'search_queue.py')
import search_queue as sq

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

OUT_DIR = ROOT / 'work' / 'outputs'
CACHE = OUT_DIR / 'cache'            # parquet frames, gitignored: reruns skip the warehouse scans
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_DATES = sq.decision_dates('2026-01-05', '2026-05-11')     # every second Monday; outcomes end 2026-05-24
DEV_MONTHS = sq.months_needed(DEV_DATES)
assert '2026-06' not in DEV_MONTHS, 'June 2026 is the sealed month: development never opens it'

con = sq.open_warehouse(DEV_MONTHS, HF_TOKEN)
frames = sq.build_frames(con, DEV_DATES, cache_dir=CACHE)
dev = frames[frames['label_ok']].reset_index(drop=True)
SEED = 42

print(f'{len(DEV_DATES)} decision dates {DEV_DATES[0].date()} -> {DEV_DATES[-1].date()} | months {DEV_MONTHS[0]} -> {DEV_MONTHS[-1]}')
print(f'{len(frames):,} page-date rows; {len(dev):,} with a complete outcome window; '
      f'{dev.client_hash_id.nunique()} clients | pandas {pd.__version__}, scikit-learn {sklearn.__version__}')

10 decision dates 2026-01-05 -> 2026-05-11 | months 2025-12 -> 2026-05
659,482 page-date rows; 654,186 with a complete outcome window; 50 clients | pandas 2.3.3, scikit-learn 1.7.2


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper (*The State of AI-Driven SEO in Numbers*, March 2026; `docs/flyrank-seo-research-march-2026.pdf`)
is careful in many places. It leads with direct comparisons, calls its ML pages exploratory, and says
outright that it is observational. My questions are about two findings that point straight at my lane,
because they tell an editor *which pages to refresh*.

**Finding #1 — "The Anatomy of Growing Content" (CONFIRMED).** Growing pages average 184 days old,
declining pages 230 days. The advice: *"review aging pages before they drift into decline"*. The
appendix backs it with a logistic regression (*"71% holdout accuracy"*, 80/20 split) in which
*"content age is the strongest negative signal"*.
- *Where does the label come from?* `trend_direction` compares the last 30 days with the 30 days
  before. Age and the trend are measured over the **same** window, so it is a snapshot comparison,
  not a forecast.
- *Does the validation carry the claim?*
  - An 80/20 split over pages lets pages of the same brand sit on both sides.
  - 71% accuracy is reported without the base rate.
  - Pooled averages can mix brands of different ages (a brand that joined later has younger pages
    *and* its own trend).
- *My check:* a forward label (the 14 days *after* T) at ten decision dates, read both pooled and
  **inside each client**.

**Finding #4 — "The Freshness Multiplier" (CONFIRMED).** Pages older than 365 days that were refreshed
within 30 days show a *"3.2× health boost"* and *"57× more impressions"*. The paper calls refresh timing
*"one of the strongest measured levers available"*.
- *Where does the label come from?* "Freshness" is days since the last content update. The comparison
  is refreshed vs not-refreshed pages, observed after the fact.
- *Does the design carry the claim?* A multiplier between two groups only reads as a *lever* if the
  groups were comparable before the refresh. There's no before/after on the same pages and no control
  for which pages got chosen.
- *My check:*
  1. What do the warehouse's update and optimisation dates actually record?
  2. Were the later-optimised pages *already* different before anyone touched them?

In [2]:
# ---- Finding #1: is "older pages decline" visible with a forward label, pooled and within clients?
age_band = pd.cut(dev['age_days_at_T'], [0, 90, 180, 270, 365, np.inf], right=False,
                  labels=['0-89 d', '90-179 d', '180-269 d', '270-364 d', '365+ d'])
pooled = dev.groupby(age_band, observed=True)['declined_robust'].agg(pages='size', decline_rate='mean')
print('Forward 2-week decline rate by page age (10 decision dates, pooled):')
print(pooled.round(3).to_string(), '\n')

mean_age = dev.groupby('declined_robust')['age_days_at_T'].mean()
print(f"mean age, pooled: declined {mean_age[1]:.0f} d vs not declined {mean_age[0]:.0f} d")

# the same comparison INSIDE each client at each date (cells with >= 200 pages and both outcomes)
def age_gap(g):
    return g.loc[g['declined_robust'] == 1, 'age_days_at_T'].mean() - g.loc[g['declined_robust'] == 0, 'age_days_at_T'].mean()
cells = dev.groupby(['decision_date', 'client_hash_id']).filter(lambda g: len(g) >= 200 and g['declined_robust'].nunique() == 2)
gaps = cells.groupby(['decision_date', 'client_hash_id']).apply(age_gap, include_groups=False)
print(f"within client-dates (n={len(gaps)}): declined pages are OLDER in {(gaps > 0).sum()}, YOUNGER in {(gaps < 0).sum()}; "
      f"median gap {gaps.median():+.0f} days\n")

# ---- Finding #4: what do the update / optimisation dates record, and who gets refreshed?
dates_dc = con.sql('SELECT content_hash_id, content_updated_date, last_optimized_date FROM dim_content').df()
m = dev[dev['decision_date'] == pd.Timestamp('2026-03-16')].merge(dates_dc, on='content_hash_id', how='left')
upd = pd.to_datetime(m['content_updated_date'])
top_days = upd.dt.date.value_counts(normalize=True).head(3)
per_client_top_day = m.assign(d=upd.dt.date).groupby('client_hash_id')['d'].agg(lambda s: s.value_counts(normalize=True).iloc[0])
print('content_updated_date, pages scored at T = 2026-03-16:')
print(f"  share whose update date is AFTER T: {(upd >= pd.Timestamp('2026-03-16')).mean():.3f}")
print('  three most common update days:', {str(k): round(v, 3) for k, v in top_days.items()})
print(f"  inside a typical client, {per_client_top_day.median():.0%} of pages share ONE update day (median over {len(per_client_top_day)} clients)")
print(f"  the export date 2026-07-03 alone: {(upd.dt.date == pd.Timestamp('2026-07-03').date()).mean():.1%} of pages\n")

m['optimised_after_T'] = pd.to_datetime(m['last_optimized_date']) >= pd.Timestamp('2026-03-16')
selection = m.groupby('optimised_after_T').agg(pages=('imp_pre', 'size'), median_impressions_before_T=('imp_pre', 'median'),
                                                median_position=('pos_pre', 'median'), decline_rate_next_2w=('declined_robust', 'mean'))
print('Pages optimised AFTER T vs not — measured BEFORE any of that optimisation:')
selection.round(3)

Forward 2-week decline rate by page age (10 decision dates, pooled):
                pages  decline_rate
age_days_at_T                      
0-89 d         169464         0.367
90-179 d       153583         0.325
180-269 d      161458         0.352
270-364 d      101597         0.311
365+ d          68084         0.321 

mean age, pooled: declined 186 d vs not declined 194 d


within client-dates (n=171): declined pages are OLDER in 81, YOUNGER in 90; median gap -1 days



content_updated_date, pages scored at T = 2026-03-16:
  share whose update date is AFTER T: 0.802
  three most common update days: {'2026-02-25': 0.191, '2026-05-20': 0.133, '2026-07-03': 0.103}
  inside a typical client, 76% of pages share ONE update day (median over 36 clients)
  the export date 2026-07-03 alone: 10.3% of pages

Pages optimised AFTER T vs not — measured BEFORE any of that optimisation:


,pages,median_impressions_before_T,median_position,decline_rate_next_2w
optimised_after_T,,,,
False,41465,336.0,7.640,0.395
True,33077,1137.0,7.163,0.255


Both checks run on the internship warehouse release, a related pseudonymized FlyRank dataset, not the paper's exact March snapshot. So they test whether each finding *travels*, not whether the paper computed it correctly.

**Finding #1 — with a forward label, age does not separate decliners.**
- **Pooled:** the 2-week decline rate by age band is 0.367 (0–89 days), 0.325, 0.352, 0.311 and 0.321 (365+ days). There is no rise with age, and the *youngest* band declines most.
- **Average age:** pages that went on to decline average 186 days, against 194 days for pages that held. They are slightly *younger*, the opposite direction to the paper's 230 vs 184 days.
- **Inside each client** (171 client-dates with ≥ 200 pages): declined pages are older in 81 and younger in 90, with a median gap of −1 day. That is a coin flip.
- **Suggestion for the paper:** recompute Finding #1 with the trend measured *after* the age snapshot, compare inside each brand, and report the base rate next to the 71% holdout accuracy with a brand-grouped split. If age still separates growing from declining pages there, the claim gets much stronger.

**Finding #4 — the update dates record system activity, and refreshed pages were different beforehand.**
- **What the dates are:** 80.2% of the pages scored on 2026-03-16 carry an update date *after* that day. One day alone (2026-02-25) holds 19.1% of all update dates, and 10.3% of pages carry the export date itself. Inside a typical client, **76%** of pages share a single update day. A column like that records bulk re-saves and syncs, not editorial refreshes, so a "days since update" freshness built on it measures system activity.
- **Who gets optimised:** pages that were optimised after T already had **3.4× the impressions before any of that work** (median 1,137 vs 336) and declined less over the next two weeks (0.255 vs 0.395). The pages chosen for attention were already the stronger ones, so a refreshed-vs-untouched multiplier such as 57× mixes *selection* with whatever the refresh did.
- **Suggestion for the paper:** a before/after on the same pages against a matched untouched group (same age, visibility and pre-trend), with refresh dates taken from an editorial log rather than a system timestamp. If the multiplier survives that, it becomes the strongest claim in the paper.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model is logistic regression on the five ML-04 features with the contract label `declined`.
In ML-08 I scored it on **one** client split at **one** date (precision@50 = 0.96). Here it runs on all
ten dates under three designs, from careless to honest. Each design is scored the same way: a 50-page
queue per (decision date × test fold), then mean ± SD over those cells.

| Design | Who can be on both sides | Time |
|---|---|---|
| random rows | same client, same date | the future can train the past |
| client-grouped | nobody | the future can still train the past |
| **rolling-origin + client-grouped** | nobody | train only on earlier dates |

Then the classic mistake on purpose: give a gradient-boosting model the **client code and the date
code** as features. Under a careless split it can memorise each site's and each week's base rate.

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
hgb = lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=SEED)
ev = dev[dev['decision_date'] >= DEV_DATES[2]].reset_index(drop=True)    # the dates the rolling design can test

def careless(fold_of, features, make, label):
    rows = []
    for f in range(5):
        tr, te = ev[fold_of != f], ev[fold_of == f]
        te = te.assign(p=make().fit(tr[features], tr[label]).predict_proba(te[features])[:, 1])
        for _, cell in te.groupby('decision_date'):
            rows.append(sq.queue_metrics(cell, cell['p'].to_numpy(), label, 'gap'))
    return pd.DataFrame(rows)

random_rows = np.zeros(len(ev), int)
for i, (_, te_idx) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(ev)):
    random_rows[te_idx] = i
client_grouped = ev['client_hash_id'].map(sq.client_folds(ev['client_hash_id'])).to_numpy()

ev['client_code'] = ev['client_hash_id'].astype('category').cat.codes      # the mistake, on purpose
ev['date_code'] = ev['decision_date'].rank(method='dense')
dev_ids = dev.assign(client_code=dev['client_hash_id'].astype('category').cat.codes,
                     date_code=dev['decision_date'].rank(method='dense'))
ID_FEATURES = sq.CONTRACT_FEATURES + ['client_code', 'date_code']

results = {}
for model_name, features, make, label in [('Week-5 model: LR, 5 features', sq.CONTRACT_FEATURES, lr, 'declined'),
                                          ('HGB + client code + date code', ID_FEATURES, hgb, 'declined_robust')]:
    rolling = sq.cell_metrics(sq.rolling_backtest(dev_ids, {'m': (features, make)}, label=label), {'m': 'p_m'}, label, 'gap')
    for split_name, table in [('random rows', careless(random_rows, features, make, label)),
                              ('client-grouped', careless(client_grouped, features, make, label)),
                              ('rolling-origin + client-grouped', rolling)]:
        results[(model_name, split_name)] = table[['P@50', 'AP', 'AUC', 'base_rate']].agg(['mean', 'std']).T.stack()
boundary = pd.DataFrame(results).T.round(3)
boundary.columns = [f'{a} {b}' for a, b in boundary.columns]
boundary

P@50 mean  \
Week-5 model: LR, 5 features  random rows                          0.895   
                              client-grouped                       0.882   
                              rolling-origin + client-grouped      0.852   
HGB + client code + date code random rows                          0.960   
                              client-grouped                       0.822   
                              rolling-origin + client-grouped      0.752   

                                                               P@50 std  \
Week-5 model: LR, 5 features  random rows                         0.061   
                              client-grouped                      0.094   
                              rolling-origin + client-grouped     0.143   
HGB + client code + date code random rows                         0.049   
                              client-grouped                      0.134   
                              rolling-origin + client-grouped     0.148   

                                                               AP mean  \
Week-5 model: LR, 5 features  random rows                        0.520   
                              client-grouped                     0.510   
                              rolling-origin + client-grouped    0.554   
HGB + client code + date code random rows                        0.632   
                              client-grouped                     0.509   
                              rolling-origin + client-grouped    0.501   

                                                               AP std  \
Week-5 model: LR, 5 features  random rows                       0.117   
                              client-grouped                    0.153   
                              rolling-origin + client-grouped   0.164   
HGB + client code + date code random rows                       0.116   
                              client-grouped                    0.172   
                              rolling-origin + client-grouped   0.160   

                                                               AUC mean  \
Week-5 model: LR, 5 features  random rows                         0.657   
                              client-grouped                      0.649   
                              rolling-origin + client-grouped     0.658   
HGB + client code + date code random rows                         0.764   
                              client-grouped                      0.661   
                              rolling-origin + client-grouped     0.605   

                                                               AUC std  \
Week-5 model: LR, 5 features  random rows                        0.040   
                              client-grouped                     0.053   
                              rolling-origin + client-grouped    0.066   
HGB + client code + date code random rows                        0.027   
                              client-grouped                     0.064   
                              rolling-origin + client-grouped    0.069   

                                                               base_rate mean  \
Week-5 model: LR, 5 features  random rows                               0.344   
                              client-grouped                            0.341   
                              rolling-origin + client-grouped           0.381   
HGB + client code + date code random rows                               0.346   
                              client-grouped                            0.342   
                              rolling-origin + client-grouped           0.383   

                                                               base_rate std  
Week-5 model: LR, 5 features  random rows                              0.118  
                              client-grouped                           0.159  
                              rolling-origin + client-grouped          0.153  
HGB + client code + date code random rows   

**The Week-5 model under three designs:**
- **ML-08** (one split, one date): precision@50 = **0.96**.
- **Random rows:** 0.895 ± 0.061.
- **Client-grouped:** 0.882 ± 0.094.
- **Rolling-origin + client-grouped:** **0.852 ± 0.143**, against a base rate of 0.381.

The single split overstated precision by about 0.1, and hid a spread of ±0.14 between dates and sites. AUC barely moves (0.657 → 0.658), because none of the five features encodes the site. **For this model the date matters more than the site boundary**: the base rate itself swings from 0.14 to 0.57 between decision dates.

**The ID mistake shows why the boundary still matters.** Give gradient boosting a client code and a date code and precision@50 reads **0.960** under random rows (AUC 0.764). Under client grouping it falls to 0.822 (AUC 0.661), and under the honest rolling-origin design to **0.752** (AUC 0.605). The careless split pays **+0.21** precision for memorising which site and which week a page belongs to, which is worth nothing on a new site or a new week. This is why the IDs stay grouping-only (`DATA_USE.md`) and why every number from here on comes from the rolling-origin design.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final setup is the five contract features, the **spike-robust label** `declined_robust` (ML-07
found that one-day spikes fooled both the rule and the plain label), and the frozen policy
*P(decline) × lost weekly impressions*. Six checks:

1. **The test suite**: window boundaries, `IS TRUE` handling, label definitions, split guards and
   metric maths, all on synthetic data.
2. **A planted leak**: the module refuses an outcome column as a feature. Bypassing the guard shows
   what the leak would have done.
3. **Shuffled labels**: a model trained on noise must fall to chance.
4. **A single-feature scan**: no single feature may look suspiciously perfect.
5. **Snapshot columns**: how often each `dim_content` date already lies after T, at every decision date.
6. **Product flags**: none exist in the release, so none can leak.

In [4]:
# 1. the test suite (skipped with a note when the notebook runs outside the repo, e.g. Colab from GitHub)
tests = ROOT / 'work' / 'tests'
if tests.is_dir() and importlib.util.find_spec('pytest'):
    run = subprocess.run([sys.executable, '-m', 'pytest', str(tests), '-q', '-p', 'no:cacheprovider', '--color=no'],
                         capture_output=True, text=True, cwd=ROOT)
    print('1. test suite:', run.stdout.strip().splitlines()[-1])
    assert run.returncode == 0, run.stdout
else:
    print('1. test suite: not available here (run `python -m pytest work/tests` from a clone)')

# 2. planted leak — the guard refuses it ...
LEAK = 'log_imp_out'
dev_leak = dev.assign(**{LEAK: np.log1p(dev['imp_out'])})
sq.OUTCOME_COLUMNS.add(LEAK)
try:
    sq.rolling_backtest(dev_leak, {'leak': (sq.CONTRACT_FEATURES + [LEAK], lr)})
    raise RuntimeError('the guard should have refused the leak')
except AssertionError as e:
    print('2. guard:', str(e))
# ... and this is what it would have done (one date, one fold, guard bypassed on purpose)
T_demo = DEV_DATES[5]
fold = dev_leak['client_hash_id'].map(sq.client_folds(dev_leak['client_hash_id']))
tr = dev_leak[(dev_leak['decision_date'] < T_demo) & (fold != 0)]
te = dev_leak[(dev_leak['decision_date'] == T_demo) & (fold == 0)]
honest_auc = sq.queue_metrics(te, lr().fit(tr[sq.CONTRACT_FEATURES], tr['declined_robust']).predict_proba(te[sq.CONTRACT_FEATURES])[:, 1])['AUC']
leaky_auc = sq.queue_metrics(te, lr().fit(tr[sq.CONTRACT_FEATURES + [LEAK]], tr['declined_robust']).predict_proba(te[sq.CONTRACT_FEATURES + [LEAK]])[:, 1])['AUC']
print(f'   bypassed on purpose at {T_demo.date()}: AUC {honest_auc:.3f} honest -> {leaky_auc:.3f} with the leak')

# 3. shuffled labels must fall to chance
shuffled = dev.assign(declined_robust=np.random.RandomState(SEED).permutation(dev['declined_robust'].to_numpy()))
noise = sq.cell_metrics(sq.rolling_backtest(shuffled, {'m': (sq.CONTRACT_FEATURES, lr)}), {'m': 'p_m'})
print(f"3. shuffled labels: AUC {noise['AUC'].mean():.3f}, precision@50 {noise['P@50'].mean():.3f} vs base rate {noise['base_rate'].mean():.3f}")

# 4. single-feature scan on the final label (|AUC - 0.5| near 0.5 would be suspicious)
from sklearn.metrics import roc_auc_score
scan = pd.Series({c: roc_auc_score(dev['declined_robust'], dev[c]) for c in sq.CONTRACT_FEATURES}).rename('AUC alone')
print('4. single-feature AUCs:', scan.round(3).to_dict())
assert (scan.sub(0.5).abs() < 0.4).all(), 'a single feature separates the label almost perfectly: investigate'

# 5. snapshot columns at every decision date
snap = dev[['decision_date', 'content_hash_id']].merge(dates_dc, on='content_hash_id', how='left')
after = snap.groupby('decision_date').apply(lambda g: pd.Series({
    'updated_after_T': (pd.to_datetime(g['content_updated_date']) >= g.name).mean(),
    'optimised_after_T': (pd.to_datetime(g['last_optimized_date']) >= g.name).mean()}), include_groups=False)
print('5. dim_content dates already AFTER T (min-max over the ten dates):',
      {c: f'{after[c].min():.2f}-{after[c].max():.2f}' for c in after})

# 6. product decision flags
flags = {'health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win', 'refresh_tier'}
columns = set(con.sql('DESCRIBE daily').df()['column_name']) | set(con.sql('DESCRIBE dim_content').df()['column_name'])
assert not flags & columns
print('6. product flags present in the release:', sorted(flags & columns) or 'none')

1. test suite: 21 passed in 5.69s


2. guard: outcome-window columns cannot be features: ['log_imp_out']


   bypassed on purpose at 2026-03-16: AUC 0.614 honest -> 0.994 with the leak


3. shuffled labels: AUC 0.499, precision@50 0.352 vs base rate 0.340


4. single-feature AUCs: {'log_imp_pre': 0.474, 'pos_pre': 0.53, 'active_days_pre': 0.496, 'momentum_pre': 0.331, 'age_days_at_T': 0.482}


5. dim_content dates already AFTER T (min-max over the ten dates): {'updated_after_T': '0.76-1.00', 'optimised_after_T': '0.43-0.53'}
6. product flags present in the release: none


**No leak in the final feature set.**
1. **Test suite:** all 21 tests pass. They cover window boundaries, `IS TRUE` handling, both label definitions, eligibility, the frozen rule's order, split guards, the outcome-feature guard, metric maths and cache round-trips.
2. **Planted leak:** the module refuses `log_imp_out` as a feature. With the guard bypassed on purpose, AUC jumps from 0.614 to **0.994**: the ML-04 lesson, now caught by code instead of by eye.
3. **Shuffled labels:** AUC 0.499 and precision@50 0.352, against a base rate of 0.340. That is chance, as it must be.
4. **Single-feature scan:** every feature alone sits between 0.33 and 0.53 AUC. The strongest is `momentum_pre` (0.331, i.e. 0.669 inverted). It is strong, not suspicious, and has a plain explanation: the slide ML-07 CONFIRMED.
5. **Snapshot columns:** `content_updated_date` already lies after T for **76–100%** of pages at every decision date, and `last_optimized_date` for 43–53%. Both stay excluded.
6. **Product flags:** none exist in the release.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

| My boldest sentence (where) | Why it needs rewriting | Safe version |
|---|---|---|
| *"The model beats the baseline: precision@50 = 0.96 vs 0.60."* (ML-08) | One date, one split; the metric rewarded tiny pages; no spread shown | "Across eight biweekly decision dates (Feb–May 2026), trained only on earlier dates and scored on clients it never saw, the Week-5 model's 50-page queues contained on average **85%** pages whose impressions then fell by more than 20% (SD 0.14; base rate 0.38). This is a **measured** result on one pseudonymized portfolio, and the top of that list is mostly small pages." |
| *"Older pages decline, so review aging pages first."* (my ML-07 starting idea; the paper's Finding #1) | My own forward test says otherwise | "In this release, page age did **not** separate pages whose visibility fell over the next two weeks from pages whose visibility held, compared inside each client across ten decision dates (**observed**)." |
| *"Refreshing old pages multiplies impressions 57×."* (paper, Finding #4) | Causal word ("lever") on a comparison between self-selected groups | "Among pages older than a year, those updated in the last 30 days showed far higher impressions than those not updated (**observed**, March 2026 snapshot). The comparison doesn't separate an effect of updating from which pages were chosen for an update." |
| *"My queue finds the pages to refresh."* (my own framing) | Implies the queue knows what fixes a page | "The queue ranks pages for **human review** by observed decline risk and visibility at stake. It is **decision-support**, and does not show that refreshing a page restores its visibility." |

The code below recomputes every number these sentences cite and writes this notebook's receipts.

In [5]:
# every number the rewritten claims cite, recomputed here, plus the receipts file
claim_numbers = {
    'week5_single_split_P@50': 0.96,                       # ML-08, one split, one date (w05_model_metrics.json)
    'week5_rolling_P@50_mean': boundary.loc[('Week-5 model: LR, 5 features', 'rolling-origin + client-grouped'), 'P@50 mean'],
    'week5_rolling_P@50_sd': boundary.loc[('Week-5 model: LR, 5 features', 'rolling-origin + client-grouped'), 'P@50 std'],
    'week5_rolling_base_rate': boundary.loc[('Week-5 model: LR, 5 features', 'rolling-origin + client-grouped'), 'base_rate mean'],
    'id_features_random_rows_P@50': boundary.loc[('HGB + client code + date code', 'random rows'), 'P@50 mean'],
    'id_features_rolling_P@50': boundary.loc[('HGB + client code + date code', 'rolling-origin + client-grouped'), 'P@50 mean'],
}
print(json.dumps({k: round(float(v), 3) for k, v in claim_numbers.items()}, indent=1))

receipts = {
    'notebook': 'work/notebooks/w06_validation_audit.ipynb', 'dev_dates': [str(d.date()) for d in DEV_DATES],
    'finding1_pooled_by_age': pooled.round(4).reset_index().astype(str).to_dict(orient='records'),
    'finding1_within_client': {'cells': int(len(gaps)), 'declined_older': int((gaps > 0).sum()),
                               'declined_younger': int((gaps < 0).sum()), 'median_gap_days': float(gaps.median())},
    'finding4_update_after_T_share': float((upd >= pd.Timestamp('2026-03-16')).mean()),
    'finding4_per_client_top_update_day_share_median': float(per_client_top_day.median()),
    'finding4_selection': selection.round(4).reset_index().astype(str).to_dict(orient='records'),
    'boundary': {f'{m} | {s}': row.to_dict() for (m, s), row in boundary.iterrows()},
    'leakage': {'planted_leak_auc': [round(honest_auc, 4), round(leaky_auc, 4)],
                'shuffled_auc': round(float(noise['AUC'].mean()), 4), 'single_feature_auc': scan.round(4).to_dict(),
                'snapshot_after_T_range': {c: [round(after[c].min(), 4), round(after[c].max(), 4)] for c in after}},
}
(OUT_DIR / 'w06_validation_receipts.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (OUT_DIR / 'w06_validation_receipts.json').relative_to(ROOT))

{
 "week5_single_split_P@50": 0.96,
 "week5_rolling_P@50_mean": 0.852,
 "week5_rolling_P@50_sd": 0.143,
 "week5_rolling_base_rate": 0.381,
 "id_features_random_rows_P@50": 0.96,
 "id_features_rolling_P@50": 0.752
}
wrote work\outputs\w06_validation_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.